# Urban Change Detection in Jeddah
## Using Sentinel-2 Imagery & Siamese U-Net Deep Learning

**Project:** Graduation Project Phase 2 (GP2)

**Objective:** Detect urban expansion in Jeddah across 2018, 2020, 2022, and 2024 using Sentinel-2 satellite imagery and a Siamese U-Net deep learning model.

### Pipeline Overview

| Step | Description |
|------|-------------|
| 1 | **Mask Generation** — Generate binary change masks from NDBI using Otsu thresholding |
| 2 | **Data Preparation** — Build PyTorch dataset with train/val/test splits and augmentation |
| 3 | **Model Training** — Train a Siamese U-Net with BCE + Dice loss |
| 4 | **Evaluation** — Compute IoU, F1, precision, recall, accuracy, and visualize results |

### Year Pairs Analyzed
- 2018 → 2020
- 2020 → 2022
- 2022 → 2024

---
## 1. Setup & Installation

In [ ]:
!pip install -q rasterio scikit-image tqdm

In [ ]:
import os
import random
import time
from glob import glob

import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Mount Google Drive & Configure Paths

Upload **only the NDBI patch folders** (~600 MB total) to Google Drive:

```
My Drive/jeddah/
└── 03_ML_Ready_Patches/
    └── NDVI NDBI NDWI NaturalColor/
        ├── 2018/Jeddah_2018_NDBI_Raw/   (~679 patches)
        ├── 2020/Jeddah_2020_NDBI_Raw/   (~678 patches)
        ├── 2022/Jeddah_2022_NDBI_Raw/   (~679 patches)
        └── 2024/Jeddah_2024_NDBI_Raw/   (~679 patches)
```

You do **not** need to upload the full rasters, visual previews, or other index patches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================= CONFIGURATION =========================
# Adjust DATA_ROOT if your folder is in a different Drive location
DATA_ROOT = "/content/drive/MyDrive/jeddah"

PATCHES_DIR = os.path.join(DATA_ROOT, "03_ML_Ready_Patches", "NDVI NDBI NDWI NaturalColor")
OUTPUT_DIR  = os.path.join(DATA_ROOT, "05_Results")
MASKS_DIR   = os.path.join(OUTPUT_DIR, "change_masks")
CKPT_DIR    = os.path.join(OUTPUT_DIR, "checkpoints")
VIS_DIR     = os.path.join(OUTPUT_DIR, "visualizations")

for d in [OUTPUT_DIR, MASKS_DIR, CKPT_DIR, VIS_DIR]:
    os.makedirs(d, exist_ok=True)

YEARS      = [2018, 2020, 2022, 2024]
YEAR_PAIRS = [(2018, 2020), (2020, 2022), (2022, 2024)]

# Training hyper-parameters
BATCH_SIZE  = 16
NUM_EPOCHS  = 50
LR          = 1e-4
WEIGHT_DECAY = 1e-5
PATIENCE    = 10
SEED        = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

---
## 3. Data Exploration

Verify the NDBI patch dataset structure and inspect sample patches.

In [ ]:
print("Dataset overview")
print("=" * 60)

for year in YEARS:
    ndbi_dir = os.path.join(PATCHES_DIR, str(year), f"Jeddah_{year}_NDBI_Raw")
    count = len(glob(os.path.join(ndbi_dir, "*.tif")))
    print(f"  {year} NDBI: {count} patches")

# Inspect a sample NDBI patch
sample = glob(os.path.join(PATCHES_DIR, "2018", "Jeddah_2018_NDBI_Raw", "*.tif"))[0]
with rasterio.open(sample) as src:
    print(f"\nSample patch: {os.path.basename(sample)}")
    print(f"  Shape : {src.shape}")
    print(f"  Bands : {src.count}")
    print(f"  Dtype : {src.dtypes}")
    print(f"  CRS   : {src.crs}")
    data = src.read(1)
    print(f"  Range : [{np.nanmin(data):.4f}, {np.nanmax(data):.4f}]")

In [ ]:
# Visualize NDBI patches across years for one location
sample_name = os.path.basename(
    glob(os.path.join(PATCHES_DIR, "2018", "Jeddah_2018_NDBI_Raw", "*.tif"))[10]
)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, year in enumerate(YEARS):
    ndbi_path = os.path.join(PATCHES_DIR, str(year), f"Jeddah_{year}_NDBI_Raw", sample_name)
    if os.path.exists(ndbi_path):
        with rasterio.open(ndbi_path) as src:
            ndbi = src.read(1)
        axes[i].imshow(ndbi, cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
        axes[i].set_title(f"NDBI {year}")
    axes[i].axis("off")

plt.suptitle(f"NDBI across years — Patch: {sample_name}", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Generate Binary Change Masks

**Method:** For each year pair, apply Otsu's threshold on NDBI to produce a binary urban mask per year, then XOR the masks to produce a binary **change** map.

- `1` = pixel changed (urban expansion or loss)
- `0` = no change

In [ ]:
def otsu_threshold(values):
    """Compute Otsu's threshold for a 1-D array of finite values."""
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return 0.0
    hist, bin_edges = np.histogram(values, bins=256)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    total = hist.sum()
    w0 = np.cumsum(hist).astype(np.float64)
    w1 = total - w0
    sum0 = np.cumsum(hist * bin_centers)
    sum_total = (hist * bin_centers).sum()
    mean0 = np.divide(sum0, w0, out=np.zeros_like(sum0), where=w0 != 0)
    mean1 = np.divide(sum_total - sum0, w1, out=np.zeros_like(w1), where=w1 != 0)
    variance = w0 * w1 * (mean0 - mean1) ** 2
    return bin_centers[np.argmax(variance)]


def generate_change_masks(patches_dir, masks_dir, year_pairs, min_valid=0.10):
    """Generate binary change masks for all year pairs."""
    summary = {}

    for t1, t2 in year_pairs:
        pair = f"{t1}_{t2}"
        print(f"\n{'=' * 60}")
        print(f"  Generating change masks: {t1} -> {t2}")
        print(f"{'=' * 60}")

        dir_t1 = os.path.join(patches_dir, str(t1), f"Jeddah_{t1}_NDBI_Raw")
        dir_t2 = os.path.join(patches_dir, str(t2), f"Jeddah_{t2}_NDBI_Raw")

        files_t1 = {os.path.basename(f): f for f in glob(os.path.join(dir_t1, "patch_*.tif"))}
        files_t2 = {os.path.basename(f): f for f in glob(os.path.join(dir_t2, "patch_*.tif"))}
        common = sorted(files_t1.keys() & files_t2.keys())
        print(f"  Common patches: {len(common)}")

        out_dir = os.path.join(masks_dir, pair)
        os.makedirs(out_dir, exist_ok=True)

        stats = {"total": 0, "with_change": 0, "skipped": 0}

        for fname in tqdm(common, desc=f"  {pair}", ncols=80):
            with rasterio.open(files_t1[fname]) as src:
                d1 = src.read(1).astype(np.float32)
                profile = src.profile.copy()
            with rasterio.open(files_t2[fname]) as src:
                d2 = src.read(1).astype(np.float32)

            valid = np.isfinite(d1) & (d1 != 0) & np.isfinite(d2) & (d2 != 0)
            if valid.mean() < min_valid:
                stats["skipped"] += 1
                continue

            # Urban masks via Otsu
            th1 = otsu_threshold(d1[valid])
            th2 = otsu_threshold(d2[valid])
            u1 = (d1 > th1).astype(np.uint8)
            u2 = (d2 > th2).astype(np.uint8)
            u1[~valid] = 0
            u2[~valid] = 0

            change = (u1 != u2).astype(np.uint8)
            change[~valid] = 0

            out_profile = profile.copy()
            out_profile.update(dtype="uint8", count=1, compress="lzw")
            with rasterio.open(os.path.join(out_dir, fname), "w", **out_profile) as dst:
                dst.write(change[np.newaxis])

            stats["total"] += 1
            if change.sum() > 0:
                stats["with_change"] += 1

        summary[pair] = stats
        print(f"  Saved {stats['total']} masks "
              f"({stats['with_change']} contain change, "
              f"{stats['skipped']} skipped)")

    return summary

In [ ]:
mask_summary = generate_change_masks(PATCHES_DIR, MASKS_DIR, YEAR_PAIRS)

print("\nSummary:")
for pair, s in mask_summary.items():
    print(f"  {pair}: {s['total']} total, {s['with_change']} with change, {s['skipped']} skipped")

In [ ]:
# Visualize a few generated change masks
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
cmap_change = ListedColormap(["black", "red"])

for row, (t1, t2) in enumerate(YEAR_PAIRS):
    pair = f"{t1}_{t2}"
    mask_dir = os.path.join(MASKS_DIR, pair)
    mask_files = sorted(glob(os.path.join(mask_dir, "*.tif")))

    # Pick a patch that has change
    sample_mask_file = None
    for mf in mask_files:
        with rasterio.open(mf) as src:
            m = src.read(1)
        if m.sum() > 500:
            sample_mask_file = mf
            break

    if sample_mask_file is None:
        sample_mask_file = mask_files[0]

    fname = os.path.basename(sample_mask_file)

    # Load NDBI patches
    with rasterio.open(os.path.join(PATCHES_DIR, str(t1), f"Jeddah_{t1}_NDBI_Raw", fname)) as src:
        ndbi_t1 = src.read(1)
    with rasterio.open(os.path.join(PATCHES_DIR, str(t2), f"Jeddah_{t2}_NDBI_Raw", fname)) as src:
        ndbi_t2 = src.read(1)
    with rasterio.open(sample_mask_file) as src:
        change_mask = src.read(1)

    axes[row, 0].imshow(ndbi_t1, cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
    axes[row, 0].set_title(f"NDBI {t1}")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(ndbi_t2, cmap="RdYlGn_r", vmin=-0.5, vmax=0.5)
    axes[row, 1].set_title(f"NDBI {t2}")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(change_mask, cmap=cmap_change, vmin=0, vmax=1)
    axes[row, 2].set_title(f"Change mask {t1}->{t2}")
    axes[row, 2].axis("off")

    # Overlay
    axes[row, 3].imshow(ndbi_t2, cmap="gray", vmin=-0.5, vmax=0.5)
    axes[row, 3].imshow(change_mask, cmap=cmap_change, alpha=0.5, vmin=0, vmax=1)
    axes[row, 3].set_title(f"Overlay {t1}->{t2}")
    axes[row, 3].axis("off")

plt.suptitle("Generated Change Masks (Red = Change)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, "change_mask_samples.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Dataset & DataLoaders

Build a PyTorch `Dataset` that loads `(NDBI_t1, NDBI_t2, change_mask)` triplets.

- **Training** set: 70% with augmentation (random flips, 90° rotations)
- **Validation** set: 15%
- **Test** set: 15%

In [ ]:
class ChangeDetectionDataset(Dataset):
    """
    Loads (NDBI_t1, NDBI_t2, change_mask) triplets.
    All tensors are float32 with shape (1, 256, 256).
    """

    def __init__(self, patches_dir, masks_dir, year_pairs, augment=False):
        self.augment = augment
        self.samples = []

        for t1, t2 in year_pairs:
            pair = f"{t1}_{t2}"
            mask_dir = os.path.join(masks_dir, pair)
            ndbi_t1 = os.path.join(patches_dir, str(t1), f"Jeddah_{t1}_NDBI_Raw")
            ndbi_t2 = os.path.join(patches_dir, str(t2), f"Jeddah_{t2}_NDBI_Raw")

            if not os.path.isdir(mask_dir):
                continue

            for mp in sorted(glob(os.path.join(mask_dir, "patch_*.tif"))):
                fname = os.path.basename(mp)
                p1 = os.path.join(ndbi_t1, fname)
                p2 = os.path.join(ndbi_t2, fname)
                if os.path.exists(p1) and os.path.exists(p2):
                    self.samples.append((p1, p2, mp))

        print(f"Dataset: {len(self.samples)} samples from {len(year_pairs)} year pair(s)")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _read(path):
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
        arr = np.nan_to_num(arr, nan=0.0)
        arr = np.clip(arr, -1.0, 1.0)
        return arr[np.newaxis]

    def _apply_aug(self, t1, t2, m):
        if random.random() > 0.5:
            t1 = np.flip(t1, axis=2).copy()
            t2 = np.flip(t2, axis=2).copy()
            m  = np.flip(m,  axis=2).copy()
        if random.random() > 0.5:
            t1 = np.flip(t1, axis=1).copy()
            t2 = np.flip(t2, axis=1).copy()
            m  = np.flip(m,  axis=1).copy()
        k = random.randint(0, 3)
        if k:
            t1 = np.rot90(t1, k, axes=(1, 2)).copy()
            t2 = np.rot90(t2, k, axes=(1, 2)).copy()
            m  = np.rot90(m,  k, axes=(1, 2)).copy()
        return t1, t2, m

    def __getitem__(self, idx):
        p1, p2, mp = self.samples[idx]
        t1 = self._read(p1)
        t2 = self._read(p2)
        with rasterio.open(mp) as src:
            mask = src.read(1).astype(np.float32)[np.newaxis]
        if self.augment:
            t1, t2, mask = self._apply_aug(t1, t2, mask)
        return torch.from_numpy(t1), torch.from_numpy(t2), torch.from_numpy(mask)


class _Subset(Dataset):
    """Wraps a subset of the base dataset with optional augmentation."""
    def __init__(self, base, indices, augment=False):
        self.base = base
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        orig = self.base.augment
        self.base.augment = self.augment
        item = self.base[self.indices[idx]]
        self.base.augment = orig
        return item

In [ ]:
full_dataset = ChangeDetectionDataset(PATCHES_DIR, MASKS_DIR, YEAR_PAIRS)

n = len(full_dataset)
indices = list(range(n))
random.seed(SEED)
random.shuffle(indices)

n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_ds = _Subset(full_dataset, indices[:n_train], augment=True)
val_ds   = _Subset(full_dataset, indices[n_train:n_train + n_val], augment=False)
test_ds  = _Subset(full_dataset, indices[n_train + n_val:], augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

# Quick sanity check
t1, t2, m = train_ds[0]
print(f"Sample shapes — t1: {t1.shape}, t2: {t2.shape}, mask: {m.shape}")

---
## 6. Siamese U-Net Model

**Architecture:**
- Two input images (NDBI_t1, NDBI_t2) pass through a **shared encoder** (weight-tied).
- Skip features from both branches are **concatenated** at each decoder level.
- A single decoder produces a binary change-probability map.

```
      ┌─ Encoder ─┐       ┌─ Encoder ─┐
t1 ──►│  (shared)  │       │  (shared)  │◄── t2
      └────┬───────┘       └────┬───────┘
           │    skip concat     │
           └────────┬──────────┘
                    ▼
              ┌──────────┐
              │ Bottleneck│
              └─────┬────┘
                    ▼
              ┌──────────┐
              │  Decoder  │
              └─────┬────┘
                    ▼
            Change Map [0,1]
```

In [ ]:
class ConvBlock(nn.Module):
    """Conv3x3-BN-ReLU x 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SiameseUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Shared encoder
        self.enc1 = ConvBlock(in_channels, 64)
        self.enc2 = ConvBlock(64, 128)
        self.enc3 = ConvBlock(128, 256)
        self.enc4 = ConvBlock(256, 512)

        # Bottleneck (concatenated features from both branches)
        self.bottleneck = ConvBlock(512 * 2, 1024)

        # Decoder: upsample + concat(skip_t1, skip_t2) + conv block
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = ConvBlock(512 + 512 * 2, 512)   # 1536 -> 512

        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ConvBlock(256 + 256 * 2, 256)   # 768 -> 256

        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ConvBlock(128 + 128 * 2, 128)   # 384 -> 128

        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ConvBlock(64 + 64 * 2, 64)      # 192 -> 64

        self.head = nn.Conv2d(64, out_channels, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x1, x2):
        # Encoder branch 1
        e1_1 = self.enc1(x1);  p1_1 = self.pool(e1_1)
        e2_1 = self.enc2(p1_1); p2_1 = self.pool(e2_1)
        e3_1 = self.enc3(p2_1); p3_1 = self.pool(e3_1)
        e4_1 = self.enc4(p3_1); p4_1 = self.pool(e4_1)

        # Encoder branch 2 (shared weights)
        e1_2 = self.enc1(x2);  p1_2 = self.pool(e1_2)
        e2_2 = self.enc2(p1_2); p2_2 = self.pool(e2_2)
        e3_2 = self.enc3(p2_2); p3_2 = self.pool(e3_2)
        e4_2 = self.enc4(p3_2); p4_2 = self.pool(e4_2)

        # Bottleneck
        b = self.bottleneck(torch.cat([p4_1, p4_2], dim=1))

        # Decoder
        d = self.dec4(torch.cat([self.up4(b),  e4_1, e4_2], dim=1))
        d = self.dec3(torch.cat([self.up3(d),  e3_1, e3_2], dim=1))
        d = self.dec2(torch.cat([self.up2(d),  e2_1, e2_2], dim=1))
        d = self.dec1(torch.cat([self.up1(d),  e1_1, e1_2], dim=1))

        return torch.sigmoid(self.head(d))


model = SiameseUNet(in_channels=1, out_channels=1).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

# Verify forward pass
with torch.no_grad():
    dummy1 = torch.randn(2, 1, 256, 256).to(DEVICE)
    dummy2 = torch.randn(2, 1, 256, 256).to(DEVICE)
    out = model(dummy1, dummy2)
    print(f"Output shape: {out.shape}  (expected: [2, 1, 256, 256])")

---
## 7. Loss Function & Training

**Loss:** Combined BCE + Dice loss — handles class imbalance well since most pixels are "no change".

**Optimizer:** Adam with weight decay.

**Scheduler:** ReduceLROnPlateau — halves LR if val loss plateaus for 5 epochs.

**Early stopping:** Stops if val loss doesn't improve for 10 epochs.

In [ ]:
class DiceBCELoss(nn.Module):
    """Weighted sum of Binary Cross-Entropy and Dice loss."""
    def __init__(self, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce = nn.BCELoss()
        self.bce_weight = bce_weight
        self.smooth = smooth

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = 1 - (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth
        )
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice


def run_epoch(model, loader, criterion, optimizer, device, is_train):
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_iou = 0.0
    n = 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for t1, t2, mask in loader:
            t1, t2, mask = t1.to(device), t2.to(device), mask.to(device)
            pred = model(t1, t2)
            loss = criterion(pred, mask)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * t1.size(0)

            p_bin = (pred > 0.5).float()
            inter = (p_bin * mask).sum(dim=(1, 2, 3))
            union = p_bin.sum(dim=(1, 2, 3)) + mask.sum(dim=(1, 2, 3)) - inter
            iou = (inter + 1e-7) / (union + 1e-7)
            total_iou += iou.sum().item()
            n += t1.size(0)

    return total_loss / n, total_iou / n

In [ ]:
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

history = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}
best_val_loss = float("inf")
wait = 0
best_path = os.path.join(CKPT_DIR, "best_model.pth")

print(f"Training for up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})")
print(f"{'='*90}")

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_iou = run_epoch(model, train_loader, criterion, optimizer, DEVICE, True)
    vl_loss, vl_iou = run_epoch(model, val_loader,   criterion, None,      DEVICE, False)

    scheduler.step(vl_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_iou"].append(tr_iou)
    history["val_iou"].append(vl_iou)

    elapsed = time.time() - t0
    lr_now = optimizer.param_groups[0]['lr']
    print(
        f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
        f"train loss {tr_loss:.4f}  IoU {tr_iou:.4f} | "
        f"val loss {vl_loss:.4f}  IoU {vl_iou:.4f} | "
        f"lr {lr_now:.1e} | {elapsed:.0f}s"
    )

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        wait = 0
        torch.save(model.state_dict(), best_path)
        print(f"  -> saved best model (val_loss={vl_loss:.4f})")
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (patience={PATIENCE})")
            break

# Reload best weights
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs, history['train_loss'], 'b-', label='Train')
ax1.plot(epochs, history['val_loss'],   'r-', label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (BCE + Dice)')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['train_iou'], 'b-', label='Train')
ax2.plot(epochs, history['val_iou'],   'r-', label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('IoU')
ax2.set_title('Training & Validation IoU')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Evaluation on Test Set

Compute pixel-level metrics:
- **Accuracy** — overall correct pixels
- **Precision** — of predicted changes, how many are real
- **Recall** — of real changes, how many are detected
- **F1-Score** — harmonic mean of precision and recall
- **IoU** — intersection over union of change class

In [ ]:
def compute_metrics(model, loader, device, threshold=0.5):
    model.eval()
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for t1, t2, mask in loader:
            t1, t2, mask = t1.to(device), t2.to(device), mask.to(device)
            pred = (model(t1, t2) > threshold).float()

            tp += (pred * mask).sum().item()
            fp += (pred * (1 - mask)).sum().item()
            fn += ((1 - pred) * mask).sum().item()
            tn += ((1 - pred) * (1 - mask)).sum().item()

    total = tp + fp + fn + tn
    precision = tp / (tp + fp + 1e-7)
    recall    = tp / (tp + fn + 1e-7)
    f1        = 2 * precision * recall / (precision + recall + 1e-7)
    iou       = tp / (tp + fp + fn + 1e-7)
    accuracy  = (tp + tn) / (total + 1e-7)

    return {
        'accuracy': accuracy, 'precision': precision,
        'recall': recall, 'f1': f1, 'iou': iou,
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
    }


test_metrics = compute_metrics(model, test_loader, DEVICE)

print("\n" + "=" * 50)
print("  TEST SET RESULTS")
print("=" * 50)
print(f"  Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"  Precision : {test_metrics['precision']:.4f}")
print(f"  Recall    : {test_metrics['recall']:.4f}")
print(f"  F1-Score  : {test_metrics['f1']:.4f}")
print(f"  IoU       : {test_metrics['iou']:.4f}")
print(f"  TP={test_metrics['tp']:,}  FP={test_metrics['fp']:,}  "
      f"FN={test_metrics['fn']:,}  TN={test_metrics['tn']:,}")
print("=" * 50)

In [ ]:
# Confusion matrix
cm = np.array([[test_metrics['tn'], test_metrics['fp']],
               [test_metrics['fn'], test_metrics['tp']]])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')

labels = ['No Change', 'Change']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Test Set)')

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,.0f}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=12)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Visual Results

Side-by-side comparison of NDBI input patches, ground truth, and model predictions.

In [ ]:
NUM_SHOW = 8
model.eval()
shown = 0

fig, axes = plt.subplots(NUM_SHOW, 4, figsize=(16, 4 * NUM_SHOW))
cmap_change = ListedColormap(['black', 'red'])

with torch.no_grad():
    for t1, t2, mask in test_loader:
        pred = model(t1.to(DEVICE), t2.to(DEVICE)).cpu()

        for i in range(t1.size(0)):
            if shown >= NUM_SHOW:
                break

            axes[shown, 0].imshow(t1[i, 0], cmap='RdYlGn', vmin=-1, vmax=1)
            axes[shown, 0].set_title('NDBI t1')
            axes[shown, 0].axis('off')

            axes[shown, 1].imshow(t2[i, 0], cmap='RdYlGn', vmin=-1, vmax=1)
            axes[shown, 1].set_title('NDBI t2')
            axes[shown, 1].axis('off')

            axes[shown, 2].imshow(mask[i, 0], cmap=cmap_change, vmin=0, vmax=1)
            axes[shown, 2].set_title('Ground Truth')
            axes[shown, 2].axis('off')

            axes[shown, 3].imshow((pred[i, 0] > 0.5).float(), cmap=cmap_change, vmin=0, vmax=1)
            axes[shown, 3].set_title('Prediction')
            axes[shown, 3].axis('off')

            shown += 1
        if shown >= NUM_SHOW:
            break

plt.suptitle('Test Set Predictions (Red = Change)', fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'test_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Generate Change Maps for Each Year Pair

Run the trained model on **all** patches for each year pair and generate complete change maps with statistics.

In [ ]:
model.eval()

for t1_year, t2_year in YEAR_PAIRS:
    pair = f"{t1_year}_{t2_year}"
    mask_dir = os.path.join(MASKS_DIR, pair)
    mask_files = sorted(glob(os.path.join(mask_dir, "*.tif")))

    ndbi_dir_t1 = os.path.join(PATCHES_DIR, str(t1_year), f"Jeddah_{t1_year}_NDBI_Raw")
    ndbi_dir_t2 = os.path.join(PATCHES_DIR, str(t2_year), f"Jeddah_{t2_year}_NDBI_Raw")

    total_change_px = 0
    total_px = 0
    pred_dir = os.path.join(OUTPUT_DIR, "predicted_masks", pair)
    os.makedirs(pred_dir, exist_ok=True)

    print(f"\nGenerating predictions for {pair} ({len(mask_files)} patches)...")

    with torch.no_grad():
        for mf in tqdm(mask_files, desc=f"  {pair}", ncols=80):
            fname = os.path.basename(mf)
            p1 = os.path.join(ndbi_dir_t1, fname)
            p2 = os.path.join(ndbi_dir_t2, fname)

            if not (os.path.exists(p1) and os.path.exists(p2)):
                continue

            with rasterio.open(p1) as src:
                d1 = np.nan_to_num(src.read(1).astype(np.float32))
                profile = src.profile.copy()
            with rasterio.open(p2) as src:
                d2 = np.nan_to_num(src.read(1).astype(np.float32))

            d1 = np.clip(d1, -1, 1)
            d2 = np.clip(d2, -1, 1)

            t1_tensor = torch.from_numpy(d1[np.newaxis, np.newaxis]).to(DEVICE)
            t2_tensor = torch.from_numpy(d2[np.newaxis, np.newaxis]).to(DEVICE)

            pred = model(t1_tensor, t2_tensor)
            pred_mask = (pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

            total_change_px += pred_mask.sum()
            total_px += pred_mask.size

            out_profile = profile.copy()
            out_profile.update(dtype='uint8', count=1, compress='lzw')
            with rasterio.open(os.path.join(pred_dir, fname), 'w', **out_profile) as dst:
                dst.write(pred_mask[np.newaxis])

    change_pct = 100.0 * total_change_px / (total_px + 1e-7)
    print(f"  {pair}: {total_change_px:,} changed pixels out of {total_px:,} "
          f"({change_pct:.2f}% change)")

In [ ]:
# Summary bar chart of change percentage per period
change_stats = {}
for t1_year, t2_year in YEAR_PAIRS:
    pair = f"{t1_year}_{t2_year}"
    pred_dir = os.path.join(OUTPUT_DIR, "predicted_masks", pair)
    total_change = 0
    total_pixels = 0
    for f in glob(os.path.join(pred_dir, "*.tif")):
        with rasterio.open(f) as src:
            m = src.read(1)
        total_change += m.sum()
        total_pixels += m.size
    change_stats[f"{t1_year}-{t2_year}"] = 100 * total_change / (total_pixels + 1e-7)

fig, ax = plt.subplots(figsize=(8, 5))
periods = list(change_stats.keys())
values = list(change_stats.values())
bars = ax.bar(periods, values, color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black')

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{v:.2f}%', ha='center', fontsize=12, fontweight='bold')

ax.set_xlabel('Time Period', fontsize=12)
ax.set_ylabel('Changed Pixels (%)', fontsize=12)
ax.set_title('Urban Change Detected per Period (Siamese U-Net)', fontsize=14)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'change_per_period.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Save Final Model & Summary

Save the trained model and a text summary of all results.

In [ ]:
# Save final model
final_path = os.path.join(CKPT_DIR, 'siamese_unet_final.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'test_metrics': test_metrics,
    'config': {
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'learning_rate': LR,
        'year_pairs': YEAR_PAIRS,
    }
}, final_path)
print(f"Model saved to: {final_path}")

# Write summary
summary_path = os.path.join(OUTPUT_DIR, 'results_summary.txt')
with open(summary_path, 'w') as f:
    f.write('=' * 60 + '\n')
    f.write('  Jeddah Urban Change Detection — Results Summary\n')
    f.write('=' * 60 + '\n\n')
    f.write(f'Model: Siamese U-Net\n')
    f.write(f'Input: NDBI patches (256x256, 1 channel)\n')
    f.write(f'Year pairs: {YEAR_PAIRS}\n')
    f.write(f'Dataset size: {len(full_dataset)} total samples\n')
    f.write(f'Train/Val/Test: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}\n')
    f.write(f'Epochs trained: {len(history["train_loss"])}\n')
    f.write(f'Best val loss: {best_val_loss:.4f}\n\n')
    f.write('Test Set Metrics:\n')
    f.write(f'  Accuracy  : {test_metrics["accuracy"]:.4f}\n')
    f.write(f'  Precision : {test_metrics["precision"]:.4f}\n')
    f.write(f'  Recall    : {test_metrics["recall"]:.4f}\n')
    f.write(f'  F1-Score  : {test_metrics["f1"]:.4f}\n')
    f.write(f'  IoU       : {test_metrics["iou"]:.4f}\n')

print(f"Summary saved to: {summary_path}")
print("\nDone! All results saved in 05_Results/")